# Task 2: Setup, Evaluation Framework, and Baseline

This notebook prepares Task 2 only. It creates the fixed season split, fits shared normalisation statistics, defines the evaluation metrics, and evaluates the majority baseline.

## How to Run

Run this notebook before the three model notebooks. Its saved split is the only split used by Task 2.

## 1. Setup


In [1]:
%matplotlib inline

import gc
import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, densenet121

from joblib import Parallel, delayed, dump as joblib_dump, load as joblib_load
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.*")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8b6fc0"]
MUTED = "#6b7280"


In [2]:
# Find the repository root whether Jupyter starts in the root or this notebook folder.
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")
sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    MANIFEST, TEST_IMAGE_DIR, IMAGE_TARGET_SIZE, compute_normalisation,
    describe_split, load_image_array, load_manifest, make_split,
)
from src.task2_utils import (
    FEATURE_CONFIG, extract_visual_features,
    TASK2_BASELINE_SCORES_PATH, TASK2_METADATA_PATH, TASK2_SPLIT_PATH,
    calculate_run_fingerprint, ensure_task2_directories,
)

ensure_task2_directories()
print("Repository root:", REPO_ROOT)
print("Manifest:", MANIFEST)
print("Image target size (w, h):", IMAGE_TARGET_SIZE)


Repository root: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2
Manifest: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\train_manifest.csv
Image target size (w, h): (60, 80)


### 1.1 Configuration

These settings control only the shared split and preprocessing. Model-specific training settings remain in each model notebook. Use the same `QUICK_RUN` value in Notebook 1 and every model notebook.


In [3]:
TARGET = "season"
RANDOM_STATE = 42
QUICK_RUN = False
VALIDATION_SHARE = 0.20
FEATURE_N_JOBS = -1
USE_EXTRA_SEASON_DATA = True
EXTRA_SEASON_ROOT = REPO_ROOT / "ExtraSeasonData" / "ExtraSeasonData"
EXTRA_SEASON_CSV = EXTRA_SEASON_ROOT / "extraSeasonData.csv"
EXTRA_SEASON_IMAGE_DIR = EXTRA_SEASON_ROOT / "extraSeasonImages"

if QUICK_RUN:
    print("QUICK_RUN: preparing a reduced training cache for workflow testing.")
else:
    print("FULL RUN: preparing all Task 2 training rows.")


FULL RUN: preparing all Task 2 training rows.


### 1.2 External Spring/Winter data audit and preparation

The original Task 2 data are strongly imbalanced, with Spring particularly underrepresented, with 4.1% coverage of all 4 seasons. This optional supplemental set adds genuinely labelled Spring and Winter garments to the **training partition only** to mitigate the imbalance class problem. It is not used for validation: keeping the original validation partition untouched preserves a consistent benchmark and prevents the different acquisition domain from making performance appear artificially better.

The images and labels come from the public Hugging Face derivative [`fnauman/fashion-second-hand-front-only-rgb`](https://huggingface.co/datasets/fnauman/fashion-second-hand-front-only-rgb), which contains the front images and annotations from *Clothing Dataset for Second-Hand Fashion, Version 3*. The derivative applies background removal and vertical orientation. The original dataset is released under CC BY 4.0 and should be cited as: Nauman, F. (2024), *Clothing Dataset for Second-Hand Fashion (Version 3)*, Zenodo, [https://doi.org/10.5281/zenodo.13788681](https://doi.org/10.5281/zenodo.13788681).

Only exact source labels `Spring` and `Winter` are accepted. Season is never inferred from appearance, filename, date, or article type. A domain shift remains possible because these are second-hand garments captured and processed differently from the original catalogue images; all model selection therefore remains based on the original validation data.


In [4]:
EXTRA_DATASET_NAME = "fnauman/fashion-second-hand-front-only-rgb"
EXTRA_DATASET_DOI = "10.5281/zenodo.13788681"
EXPECTED_EXTRA_SEASONS = {"Spring", "Winter"}
REQUIRED_EXTRA_COLUMNS = {"id", "season", "articleType"}

if USE_EXTRA_SEASON_DATA:
    if not EXTRA_SEASON_CSV.is_file():
        raise FileNotFoundError(f"Supplemental label file not found: {EXTRA_SEASON_CSV}")
    if not EXTRA_SEASON_IMAGE_DIR.is_dir():
        raise FileNotFoundError(f"Supplemental image directory not found: {EXTRA_SEASON_IMAGE_DIR}")

    extra_season_frame = pd.read_csv(EXTRA_SEASON_CSV, dtype={"id": str})
    missing_columns = REQUIRED_EXTRA_COLUMNS - set(extra_season_frame.columns)
    if missing_columns:
        raise ValueError(f"Supplemental CSV is missing columns: {sorted(missing_columns)}")

    extra_season_frame = extra_season_frame.loc[:, ["id", "season", "articleType"]].copy()
    if extra_season_frame.empty:
        raise ValueError("Supplemental CSV contains no rows.")
    if extra_season_frame[list(REQUIRED_EXTRA_COLUMNS)].isna().any().any():
        raise ValueError("Supplemental CSV contains missing required values.")
    if extra_season_frame["id"].duplicated().any():
        duplicates = extra_season_frame.loc[extra_season_frame["id"].duplicated(), "id"].tolist()
        raise ValueError(f"Supplemental CSV contains duplicate IDs: {duplicates[:10]}")

    unexpected_seasons = sorted(set(extra_season_frame["season"]) - EXPECTED_EXTRA_SEASONS)
    if unexpected_seasons:
        raise ValueError(f"Unexpected supplemental season labels: {unexpected_seasons}")

    # Preserve the source ID for provenance, while prefixing the working ID and group ID
    # so they cannot collide with the original FashionDataset identifiers.
    extra_season_frame["source_id"] = extra_season_frame["id"]
    extra_season_frame["filename"] = extra_season_frame["source_id"] + ".jpg"
    extra_season_frame["path"] = extra_season_frame["filename"].map(
        lambda filename: str(EXTRA_SEASON_IMAGE_DIR / filename)
    )
    extra_season_frame["id"] = "extra_" + extra_season_frame["source_id"]
    extra_season_frame["group_id"] = extra_season_frame["id"]
    extra_season_frame["data_source"] = "external_second_hand"

    missing_images = extra_season_frame.loc[
        ~extra_season_frame["path"].map(lambda path: Path(path).is_file()), "filename"
    ].tolist()
    labelled_filenames = set(extra_season_frame["filename"])
    available_filenames = {path.name for path in EXTRA_SEASON_IMAGE_DIR.glob("*.jpg")}
    orphan_images = sorted(available_filenames - labelled_filenames)
    if missing_images:
        raise FileNotFoundError(f"Missing supplemental images: {missing_images[:10]}")
    if orphan_images:
        raise ValueError(f"Supplemental images without CSV rows: {orphan_images[:10]}")

    extra_audit = pd.DataFrame({
        "Check": ["Label rows", "JPEG images", "Duplicate source IDs",
                  "Missing labelled images", "Orphan JPEG images", "Article types"],
        "Value": [len(extra_season_frame), len(available_filenames),
                  int(extra_season_frame["source_id"].duplicated().sum()),
                  len(missing_images), len(orphan_images),
                  extra_season_frame["articleType"].nunique()],
    })
    display(extra_audit)
    display(pd.crosstab(extra_season_frame["articleType"], extra_season_frame["season"],
                        margins=True).sort_values("All", ascending=False))
    print("Supplemental season counts:", extra_season_frame["season"].value_counts().to_dict())
    print("Prepared supplemental rows for a later training-only merge:", len(extra_season_frame))
else:
    extra_season_frame = pd.DataFrame(columns=[
        "id", "season", "articleType", "source_id", "filename",
        "path", "group_id", "data_source",
    ])
    print("External Spring/Winter augmentation is disabled.")


,Check,Value
0,Label rows,1098
1,JPEG images,1098
2,Duplicate source IDs,0
3,Missing labelled images,0
4,Orphan JPEG images,0
5,Article types,27


season,Spring,Winter,All
articleType,,,
All,563,535,1098
Jacket,119,55,174
Sweater,38,123,161
Winter Jacket,1,136,137
Top,48,26,74
Blouse,67,0,67
Trousers,45,21,66
Vest,13,49,62
Cardigan,33,14,47


Supplemental season counts: {'Spring': 563, 'Winter': 535}
Prepared supplemental rows for a later training-only merge: 1098


## 2. Data and the Evaluation Framework

This section is fixed before model training. Every candidate is evaluated on the same validation rows with the same metrics.


### 2.1 Leakage-safe split

The shared `make_split` function keeps identical-image groups on one side of the split and stratifies by `season`. Classes with too few independent groups remain in training. The class mapping is fitted from training labels and saved with the final model.


In [5]:
frame = load_manifest(TARGET)
print(f"Rows carrying an {TARGET} label: {len(frame):,}")
print(f"Distinct classes in the manifest: {frame[TARGET].nunique()}")

if QUICK_RUN:
    # Reduce the eligible population before splitting while preserving its season distribution.
    # At 80:20, 6,250 eligible rows produce 5,000 training and 1,250 validation rows.
    quick_total = min(6250, len(frame))
    if quick_total < len(frame):
        _, frame = train_test_split(
            frame, test_size=quick_total, stratify=frame[TARGET],
            random_state=RANDOM_STATE,
        )
        frame = frame.reset_index(drop=True)
    print("QUICK_RUN: eligible population reduced to", len(frame))

train_frame, val_frame = make_split(
    frame, TARGET, validation_share=VALIDATION_SHARE, random_state=RANDOM_STATE
)

display(describe_split(train_frame, val_frame, TARGET))

Rows carrying an season label: 37,826
Distinct classes in the manifest: 4


,Target,Training rows,Validation rows,Achieved validation share %,Classes in training,Classes in validation,Classes absent from validation
0,season,30260,7566,20.002115,4,4,0


#### 2.1.1 Add the external rows to training only

The original data are split before augmentation so the validation partition remains entirely from the original FashionDataset. The audited external rows are aligned to the original training-frame schema and appended only to `train_frame`. From this point onward they pass through exactly the same image loading, resizing/padding, training-only normalisation, and Random Forest feature extraction as every original training image.


In [6]:
# Preserve the untouched original split for provenance and validation guarantees.
original_train_frame = train_frame.copy()
original_val_frame = val_frame.copy()
original_train_frame["data_source"] = "original_fashion_dataset"
original_val_frame["data_source"] = "original_fashion_dataset"

if USE_EXTRA_SEASON_DATA:
    original_ids = set(frame["id"].astype(str))
    external_ids = set(extra_season_frame["id"].astype(str))
    id_collisions = sorted(original_ids & external_ids)
    if id_collisions:
        raise ValueError(f"Original/external working-ID collisions: {id_collisions[:10]}")

    unknown_labels = sorted(set(extra_season_frame[TARGET]) - set(frame[TARGET].dropna()))
    if unknown_labels:
        raise ValueError(f"External labels absent from the original label space: {unknown_labels}")

    # Reindexing gives the external rows precisely the same columns and order as the
    # original training frame. Unavailable catalogue-only metadata remains NaN and is
    # never used as a Task 2 model input.
    training_columns = [*original_train_frame.columns]
    extra_training_frame = extra_season_frame.reindex(columns=training_columns).copy()
    train_frame = pd.concat(
        [original_train_frame, extra_training_frame], ignore_index=True, sort=False
    )
else:
    extra_training_frame = extra_season_frame.reindex(columns=original_train_frame.columns).copy()
    train_frame = original_train_frame.copy()

val_frame = original_val_frame
assert list(train_frame.columns) == list(val_frame.columns)
assert train_frame["id"].astype(str).is_unique
assert val_frame["id"].astype(str).is_unique
assert set(train_frame["id"].astype(str)).isdisjoint(set(val_frame["id"].astype(str)))
assert train_frame["path"].map(lambda path: Path(path).is_file()).all()
assert val_frame["data_source"].eq("original_fashion_dataset").all()

augmentation_summary = pd.DataFrame([
    {"Partition": "Original training", "Rows": len(original_train_frame)},
    {"Partition": "External training addition", "Rows": len(extra_training_frame)},
    {"Partition": "Final augmented training", "Rows": len(train_frame)},
    {"Partition": "Original-only validation", "Rows": len(val_frame)},
])
display(augmentation_summary)
display(pd.DataFrame({
    "Original training": original_train_frame[TARGET].value_counts(),
    "External addition": extra_training_frame[TARGET].value_counts(),
    "Augmented training": train_frame[TARGET].value_counts(),
    "Original validation": val_frame[TARGET].value_counts(),
}).fillna(0).astype(int))


,Partition,Rows
0,Original training,30260
1,External training addition,1098
2,Final augmented training,31358
3,Original-only validation,7566


,Original training,External addition,Augmented training,Original validation
season,,,,
Fall,8208,0,8208,2052
Spring,1247,563,1810,312
Summer,14928,0,14928,3732
Winter,5877,535,6412,1470


In [7]:
# Label encoding. Fixed to the sorted training classes and exported for all model notebooks, because
# reconstructing it later from a different frame would silently permute every prediction.
CLASSES = sorted(train_frame[TARGET].unique())

CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)

# Any validation class absent from training cannot be predicted. make_split sends
# single-group classes to training, so this should be empty; the check is what proves it.
unseen = sorted(set(val_frame[TARGET]) - set(CLASSES))
assert not unseen, f"Validation holds classes never seen in training: {unseen}"

y_train = train_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()
y_val = val_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()

train_support = pd.Series(np.bincount(y_train, minlength=N_CLASSES), index=CLASSES)
val_support = pd.Series(np.bincount(y_val, minlength=N_CLASSES), index=CLASSES)
SCOREABLE = np.flatnonzero(val_support.to_numpy() > 0)   # class indices macro averages use

print(f"Classes: {N_CLASSES}")
print(f"Scoreable in validation: {len(SCOREABLE)} | absent: {N_CLASSES - len(SCOREABLE)}")
print(f"Training support range: {train_support.max():,} down to {train_support.min()}")
print("Absent from validation:", sorted(np.array(CLASSES)[val_support.to_numpy() == 0]))

Classes: 4
Scoreable in validation: 4 | absent: 0
Training support range: 14,928 down to 1810
Absent from validation: []


In [8]:
class_table = pd.DataFrame({
    "Training images": train_support,
    "Validation images": val_support,
})
class_table["Training share %"] = class_table["Training images"] / len(y_train) * 100
display(class_table.style.format({"Training share %": "{:.1f}%"}))


,Training images,Validation images,Training share %
Fall,8208,2052,26.2%
Spring,1810,312,5.8%
Summer,14928,3732,47.6%
Winter,6412,1470,20.4%


### 2.2 Loading images into memory

The images are decoded once using the deterministic transform from notebook 00 and retained as `uint8`. Training augmentation is still sampled separately for every batch.


In [9]:
DEVICE = torch.device("cpu")  # setup does not need the GPU


In [10]:
def build_image_cache(frame, description):
    """Decode a frame's images once through the shared transform into one uint8 array.

    Only the deterministic transform from Section 3.1 of notebook 00 is applied here, exactly
    once per image. Augmentation still happens per epoch, so nothing about the training
    distribution is frozen by this cache.

    Returns:
        Array of shape (rows, height, width, 3), dtype uint8, in the frame's row order.
    """
    width, height = IMAGE_TARGET_SIZE
    images = np.empty((len(frame), height, width, 3), dtype=np.uint8)
    start = time.time()
    for position, path in enumerate(frame["path"]):
        images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)
        if position and position % 10000 == 0:
            print(f"  {description}: {position:,} / {len(frame):,}")
    print(f"{description}: {len(frame):,} images in {time.time() - start:.0f}s "
          f"({images.nbytes / 1e6:.0f} MB)")
    return images


def report_memory(label=""):
    """Host RSS and, on CUDA, device allocation. Cheap, and it makes a leak visible early."""
    line = []
    try:
        import resource
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        line.append(f"host peak {peak_kb / 1e6:.2f} GB")
    except (ImportError, AttributeError):
        try:
            import psutil
            line.append(f"host RSS {psutil.Process().memory_info().rss / 1e9:.2f} GB")
        except ImportError:
            pass
    if DEVICE.type == "cuda":
        line.append(f"device allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB "
                    f"reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"[memory{' ' + label if label else ''}] " + " | ".join(line))


X_train_images = build_image_cache(train_frame, "train")
X_val_images = build_image_cache(val_frame, "validation")

assert len(X_train_images) == len(y_train) and len(X_val_images) == len(y_val)
report_memory("after caching")

  train: 10,000 / 31,358
  train: 20,000 / 31,358
  train: 30,000 / 31,358
train: 31,358 images in 337s (452 MB)
validation: 7,566 images in 79s (109 MB)
[memory after caching] host RSS 1.36 GB


### 2.3 Training-only normalisation

RGB mean and standard deviation are fitted on the Task 2 training rows only. The same constants are then applied to validation and test images.


In [11]:
VERIFY_NORMALISATION = False   # True: re-decode from disk and assert the constants match

start = time.time()
# Sums are taken over the raw 0-255 values in float32 and divided by 255 at the end, which is
# the same statistic as scaling first: sum(x/255) is sum(x)/255, and the same for the squares.
# The reductions accumulate into float64, so the running totals stay exact at this scale while
# the working chunk stays float32. Promoting the chunk itself to float64 would cost four bytes
# per channel per pixel twice over, once for the chunk and once for its square.
total = np.zeros(3, dtype=np.float64)
total_square = np.zeros(3, dtype=np.float64)
n_pixels = 0
for begin in range(0, len(X_train_images), 2048):
    chunk = X_train_images[begin:begin + 2048].astype(np.float32)
    total += chunk.sum(axis=(0, 1, 2), dtype=np.float64)
    total_square += np.einsum("nhwc,nhwc->c", chunk, chunk, dtype=np.float64)
    n_pixels += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]

mean_raw = total / n_pixels
variance_raw = np.maximum(total_square / n_pixels - mean_raw ** 2, 0.0)
NORM_MEAN = (mean_raw / 255.0).astype(np.float32)
NORM_STD = np.maximum(np.sqrt(variance_raw) / 255.0, 1e-6).astype(np.float32)

print(f"Fitted on {len(train_frame):,} training rows in {time.time() - start:.1f}s "
      "(from the cache, no second decode pass)")
print("Mean (R, G, B):", np.round(NORM_MEAN, 4))
print("Std  (R, G, B):", np.round(NORM_STD, 4))

if VERIFY_NORMALISATION:
    reference_mean, reference_std = compute_normalisation(train_frame,
                                                          target_size=IMAGE_TARGET_SIZE)
    print("Reference mean:", np.round(reference_mean, 4))
    print("Reference std: ", np.round(reference_std, 4))
    assert np.allclose(NORM_MEAN, reference_mean, atol=1e-4), "Mean disagrees with notebook 00"
    assert np.allclose(NORM_STD, reference_std, atol=1e-4), "Std disagrees with notebook 00"
    print("Verified against compute_normalisation.")

# White studio backgrounds dominate, so a mean near 0.9 is the expected result rather than a bug.
assert (NORM_MEAN > 0.5).all(), "Unexpectedly dark mean; check the transform before continuing."

Fitted on 31,358 training rows in 6.6s (from the cache, no second decode pass)
Mean (R, G, B): [0.8512 0.8346 0.829 ]
Std  (R, G, B): [0.2709 0.2824 0.286 ]


### 2.4 Random Forest feature preprocessing

Random Forest cannot learn directly from image tensors. The shared visual feature extractor converts every prepared image into colour, texture, edge and foreground descriptors once. The saved matrices are reused by Notebook 2, so training does not repeat this work.


In [12]:
def build_feature_cache(images, description):
    print(f"Extracting {description} visual features...")
    features = np.vstack(Parallel(n_jobs=FEATURE_N_JOBS)(
        delayed(extract_visual_features)(image) for image in images
    )).astype(np.float32)
    print(f"{description}: {features.shape[0]:,} rows x {features.shape[1]:,} features")
    return features

X_train_features = build_feature_cache(X_train_images, "training")
X_val_features = build_feature_cache(X_val_images, "validation")
assert len(X_train_features) == len(y_train) and len(X_val_features) == len(y_val)
print("Feature configuration:", FEATURE_CONFIG)


Extracting training visual features...
training: 31,358 rows x 2,007 features
Extracting validation visual features...
validation: 7,566 rows x 2,007 features
Feature configuration: {'hue_bins': 12, 'saturation_bins': 8, 'value_bins': 8, 'hog_orientations': 9, 'hog_pixels_per_cell': (8, 8), 'hog_cells_per_block': (2, 2), 'foreground_threshold': 0.95}


### 2.5 Metrics

Macro-F1 is the primary metric because every season should contribute equally. Accuracy, balanced accuracy and weighted F1 provide complementary views. Top-2 accuracy measures whether the correct season appears among two suggestions; top-5 is not meaningful for this small label space.


In [13]:
def evaluate_predictions(y_true, y_pred, scores=None, name=""):
    labels = SCOREABLE
    row = {
        "Model": name,
        "Top-1 accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Balanced accuracy": recall_score(y_true, y_pred, labels=labels,
                                             average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if scores is not None:
        k = min(2, scores.shape[1])
        top_k = np.argpartition(scores, -k, axis=1)[:, -k:]
        row["Top-2 accuracy"] = np.mean([truth in choices for truth, choices in zip(y_true, top_k)])
    else:
        row["Top-2 accuracy"] = np.nan
    return pd.DataFrame([row])


def per_class_table(y_true, y_pred):
    rows = []
    for index in SCOREABLE:
        true_binary = y_true == index
        pred_binary = y_pred == index
        rows.append({
            "Season": CLASSES[index],
            "Support": int(true_binary.sum()),
            "Precision": (true_binary & pred_binary).sum() / max(pred_binary.sum(), 1),
            "Recall": (true_binary & pred_binary).sum() / max(true_binary.sum(), 1),
            "F1": f1_score(true_binary, pred_binary, zero_division=0),
        })
    return pd.DataFrame(rows)


RESULTS = []

def record(result_frame):
    RESULTS.append(result_frame)
    display(result_frame.style.format({c: "{:.4f}" for c in result_frame.columns if c != "Model"}))
    return result_frame


### 2.6 Simple baselines

The majority baseline measures what class frequency alone can achieve. The stratified-random baseline samples from the training prior and provides a second non-learning reference. Random Forest is treated as a candidate model in Section 3, not as a trivial baseline.


In [14]:
majority_index = int(np.bincount(y_train, minlength=N_CLASSES).argmax())
majority_pred = np.full_like(y_val, majority_index)
majority_scores = np.zeros((len(y_val), N_CLASSES))
majority_scores[:, majority_index] = 1.0
print(f"Majority class: {CLASSES[majority_index]} "
      f"({train_support.iloc[majority_index]:,} training images)")
record(evaluate_predictions(y_val, majority_pred, majority_scores, "Baseline: majority class"))

prior = np.bincount(y_train, minlength=N_CLASSES) / len(y_train)
rng = np.random.RandomState(RANDOM_STATE)
stratified_pred = rng.choice(N_CLASSES, size=len(y_val), p=prior)
record(evaluate_predictions(
    y_val, stratified_pred, np.tile(prior, (len(y_val), 1)), "Baseline: stratified random"
))

Majority class: Summer (14,928 training images)


,Model,Top-1 accuracy,Macro-F1,Balanced accuracy,Weighted F1,Top-2 accuracy
0,Baseline: majority class,0.4933,0.1652,0.2500,0.3259,0.6875


,Model,Top-1 accuracy,Macro-F1,Balanced accuracy,Weighted F1,Top-2 accuracy
0,Baseline: stratified random,0.3446,0.2485,0.2491,0.3489,0.7645


,Model,Top-1 accuracy,Macro-F1,Balanced accuracy,Weighted F1,Top-2 accuracy
0,Baseline: stratified random,0.344568,0.248532,0.249079,0.348884,0.764473


## 3. Export Shared Task 2 Artefacts

In [15]:
split_rows = pd.concat([
    pd.DataFrame({"id": train_frame["id"].astype(str), "split": "train",
                  "position": np.arange(len(train_frame)),
                  "data_source": train_frame["data_source"]}),
    pd.DataFrame({"id": val_frame["id"].astype(str), "split": "validation",
                  "position": np.arange(len(val_frame)),
                  "data_source": val_frame["data_source"]}),
], ignore_index=True)
TASK2_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
split_rows.to_csv(TASK2_SPLIT_PATH, index=False)

preprocessed_dir = REPO_ROOT / "preprocessed_datasets" / "task2"
preprocessed_dir.mkdir(parents=True, exist_ok=True)
def save_preprocessed_array(path, array, chunk_rows=1024):
    """Create, reuse, or atomically replace a prepared NumPy array.

    Model notebooks load these files as read-only memory maps. On Windows an open
    mapping cannot be truncated, so write the replacement beside the destination
    first and atomically swap it into place only after the write succeeds.
    """
    path = Path(path)
    if path.exists():
        existing = np.load(path, mmap_mode="r")
        compatible = existing.shape == array.shape and existing.dtype == array.dtype
        identical = compatible and all(
            np.array_equal(existing[start:start + chunk_rows], array[start:start + chunk_rows])
            for start in range(0, len(array), chunk_rows)
        )
        del existing
        if identical:
            print("Reused unchanged preprocessed array:", path.name)
            return
    existed = path.exists()
    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    temporary.unlink(missing_ok=True)
    try:
        with temporary.open("xb") as handle:
            np.save(handle, array)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    except OSError as error:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Could not replace prepared array {path}. Close any model notebook "
            "kernel that has this memory-mapped file open, then rerun this cell."
        ) from error
    print("Replaced preprocessed array:" if existed else "Saved preprocessed array:", path.name)

save_preprocessed_array(preprocessed_dir / "task2_deep_learning_train_images.npy", X_train_images)
save_preprocessed_array(preprocessed_dir / "task2_deep_learning_validation_images.npy", X_val_images)
save_preprocessed_array(preprocessed_dir / "task2_random_forest_train_features.npy", X_train_features)
save_preprocessed_array(preprocessed_dir / "task2_random_forest_validation_features.npy", X_val_features)

fingerprint = calculate_run_fingerprint(
    target=TARGET, classes=CLASSES, train_ids=train_frame["id"],
    validation_ids=val_frame["id"], random_state=RANDOM_STATE,
    image_target_size=IMAGE_TARGET_SIZE, normalisation_mean=NORM_MEAN,
    normalisation_std=NORM_STD, feature_version=FEATURE_CONFIG,
)
config = {
    "target": TARGET, "classes": CLASSES, "random_state": RANDOM_STATE, "quick_run": QUICK_RUN,
    "validation_share": VALIDATION_SHARE, "image_target_size": list(IMAGE_TARGET_SIZE),
    "normalisation_mean": NORM_MEAN.tolist(), "normalisation_std": NORM_STD.tolist(),
    "feature_config": FEATURE_CONFIG, "feature_count": int(X_train_features.shape[1]),
    "fingerprint": fingerprint,
    "external_data": {
        "enabled": USE_EXTRA_SEASON_DATA,
        "dataset": EXTRA_DATASET_NAME,
        "doi": EXTRA_DATASET_DOI,
        "training_only": True,
        "rows": int(len(extra_training_frame)),
        "season_counts": {
            str(label): int(count)
            for label, count in extra_training_frame[TARGET].value_counts().items()
        },
    },
    "original_training_count": int(len(original_train_frame)),
    "augmented_training_count": int(len(train_frame)),
    "train_ids": train_frame["id"].astype(str).tolist(),
    "validation_ids": val_frame["id"].astype(str).tolist(),
    "y_train": y_train.astype(int).tolist(),
    "y_validation": y_val.astype(int).tolist(),
}
TASK2_METADATA_PATH.write_text(json.dumps(config, indent=2))
with TASK2_BASELINE_SCORES_PATH.open("wb") as handle:
    np.savez_compressed(
        handle, fingerprint=fingerprint, validation_ids=val_frame["id"].astype(str),
        true_indices=y_val, classes=np.asarray(CLASSES),
        majority_scores=majority_scores,
    )
print("Saved split:", TASK2_SPLIT_PATH)
print("Run fingerprint:", fingerprint)
print("Saved metadata:", TASK2_METADATA_PATH)
print("Saved baseline evidence:", TASK2_BASELINE_SCORES_PATH)

print("Saved preprocessed arrays:", preprocessed_dir)


Replaced preprocessed array: task2_deep_learning_train_images.npy
Reused unchanged preprocessed array: task2_deep_learning_validation_images.npy
Replaced preprocessed array: task2_random_forest_train_features.npy
Reused unchanged preprocessed array: task2_random_forest_validation_features.npy
Saved split: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\splits\task2_season_split.csv
Run fingerprint: c88d2772c606
Saved metadata: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\task2\task2_metadata.json
Saved baseline evidence: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\task2\task2_baseline_validation_scores.npz
Saved preprocessed arrays: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\task2
